# LIGO GW150914: an hour of gravitational-wave strain

GWOSC publishes 4096-second strain files at 4096 Hz and 16384 Hz.
That is 16.8 million or 67.1 million samples in a single line. At the
full-hour overview GW150914 is invisible in the noise; zoom around
`t = 0` and XY refines the decimated line until the chirp appears.

The default 4 kHz HDF5 file is about 134 MB. Set
`LIGO_SAMPLE_RATE=16384` for the full-rate, roughly 536 MB file.

**Source:** [GWOSC GW150914 event page](https://gwosc.org/events/GW150914/)
and [GWOSC URL lookup documentation](https://gwosc.readthedocs.io/en/stable/locate.html).
GWOSC event data are released under CC BY 4.0; follow the
acknowledgement guidance linked from the event page.

Install beside XY with
`python -m pip install numpy requests h5py gwosc xy`.


In [ ]:
import os
from pathlib import Path

import h5py
import numpy as np
import requests
from gwosc.locate import get_event_urls

import xy

DATA_DIR = Path(os.getenv("XY_REAL_WORLD_DATA", "data")) / "gwosc"
DATA_DIR.mkdir(parents=True, exist_ok=True)

EVENT = "GW150914"
EVENT_GPS = 1_126_259_462.4
DETECTOR = os.getenv("LIGO_DETECTOR", "H1")
SAMPLE_RATE = int(os.getenv("LIGO_SAMPLE_RATE", "4096"))
DURATION = 4096
if SAMPLE_RATE not in {4096, 16384}:
    raise ValueError("LIGO_SAMPLE_RATE must be 4096 or 16384")

urls = get_event_urls(
    EVENT,
    catalog="GWTC-1-confident",
    version=3,
    detector=DETECTOR,
    duration=DURATION,
    sample_rate=SAMPLE_RATE,
    format="hdf5",
)
if not urls:
    raise RuntimeError("GWOSC returned no matching strain file")
url = urls[0]
hdf5_path = DATA_DIR / Path(url).name
if not hdf5_path.exists():
    with requests.get(
        url,
        stream=True,
        timeout=(30, 3600),
    ) as response:
        response.raise_for_status()
        partial = hdf5_path.with_suffix(".hdf5.part")
        with partial.open("wb") as output:
            for chunk in response.iter_content(chunk_size=4 * 1024 * 1024):
                output.write(chunk)
        partial.replace(hdf5_path)

print(f"cached strain file: {hdf5_path}")

In [ ]:
with h5py.File(hdf5_path, "r") as data:
    strain = np.asarray(data["strain"]["Strain"], dtype=np.float64)
    gps_start = float(np.asarray(data["meta"]["GPSstart"]))
    x_spacing = float(
        data["strain"]["Strain"].attrs.get(
            "Xspacing",
            1 / SAMPLE_RATE,
        )
    )

seconds_from_event = gps_start + np.arange(strain.size, dtype=np.float64) * x_spacing - EVENT_GPS
print(
    f"{strain.size:,} samples · {1 / x_spacing:,.0f} Hz · "
    f"{strain.nbytes / 2**20:,.1f} MiB canonical strain"
)

In [ ]:
chart = xy.line_chart(
    xy.line(
        seconds_from_event,
        strain,
        name=f"{DETECTOR} strain",
        color="#67e8f9",
        width=1.25,
    ),
    xy.vline(
        0,
        text="GW150914",
        color="#fb7185",
        width=2,
        style={"dash": "6,5"},
    ),
    xy.x_axis(
        label="seconds from GW150914",
        bounds=(
            float(seconds_from_event[0]),
            float(seconds_from_event[-1]),
        ),
    ),
    xy.y_axis(label="dimensionless strain h(t)"),
    xy.legend(),
    xy.theme(
        background="#020617",
        text_color="#e2e8f0",
        grid_color="#1e293b",
        axis_color="#94a3b8",
    ),
    title=(f"GWOSC {DETECTOR} · {strain.size:,} samples · zoom around t = 0"),
    width=1150,
    height=560,
)
payload = chart.figure().build_payload()[0]
print("render tier:", payload["traces"][0]["tier"])
chart